#  Notebook 5: CLV Calculation & Revenue at Risk

*Customer Lifetime Value · Revenue at Risk quantification · Actionable recommendations*

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import os, json, warnings
warnings.filterwarnings('ignore')

os.makedirs('../outputs', exist_ok=True)
churn_df = pd.read_csv('../outputs/churn_predictions.csv')
rfm_df   = pd.read_csv('../outputs/rfm_segments.csv')

df = churn_df.merge(rfm_df[['customer_id','Segment']].drop_duplicates('customer_id'),
                    on='customer_id', how='left', suffixes=('','_rfm'))
if 'Segment' not in df.columns and 'Segment_rfm' in df.columns:
    df['Segment'] = df['Segment_rfm']



## 1. CLV Calculation

In [2]:
OBSERVATION_MONTHS  = 18   # Sep 2016 – Mar 2018
LIFESPAN_MONTHS     = 24   # industry standard 2-yr lifespan

df['avg_order_value']    = df['total_payment_value'] / df['num_orders'].clip(lower=1)
df['purchase_frequency'] = df['num_orders'] / OBSERVATION_MONTHS
df['CLV']                = df['avg_order_value'] * df['purchase_frequency'] * LIFESPAN_MONTHS

print("=== CLV Summary ===")
print(df['CLV'].describe().round(2))
print(f"\nTotal CLV across all customers: R$ {df['CLV'].sum():,.0f}")


=== CLV Summary ===
count    96469.00
mean       214.11
std        292.88
min         12.79
25%         82.77
50%        140.63
75%        236.17
max      18218.77
Name: CLV, dtype: float64

Total CLV across all customers: R$ 20,654,633


## 2. Revenue at Risk

In [3]:
df['revenue_at_risk'] = df['CLV'] * df['churn_prob']

total_clv     = df['CLV'].sum()
total_at_risk = df['revenue_at_risk'].sum()
high_risk_n   = (df['churn_prob'] > 0.7).sum()

print(f"Total CLV (all customers)   : R$ {total_clv:>12,.0f}")
print(f"Total Revenue at Risk       : R$ {total_at_risk:>12,.0f}")
print(f"   {total_at_risk/total_clv*100:.1f}% of total CLV is at risk")
print(f"High-Risk customers (p>0.7) : {high_risk_n:>12,}")


Total CLV (all customers)   : R$   20,654,633
Total Revenue at Risk       : R$          654
   0.0% of total CLV is at risk
High-Risk customers (p>0.7) :            0


## 3. CLV Distribution by Segment

In [4]:
fig_box = px.box(df[df['CLV'] < df['CLV'].quantile(0.99)],  # trim outliers for viz
    x='Segment', y='CLV', color='Segment',
    color_discrete_sequence=px.colors.qualitative.Bold,
    title=' CLV Distribution by Customer Segment')
fig_box.update_layout(template='plotly_dark', font_family='Inter', showlegend=False)
fig_box.show()


## 4. Revenue at Risk by Segment

In [5]:
seg_risk = df.groupby('Segment').agg(
    customers      =('customer_id','count'),
    total_clv      =('CLV','sum'),
    revenue_at_risk=('revenue_at_risk','sum'),
    avg_churn_prob =('churn_prob','mean')
).sort_values('revenue_at_risk', ascending=False).reset_index()

print(seg_risk.to_string(index=False))

fig_risk = px.bar(seg_risk, x='Segment', y='revenue_at_risk',
    color='avg_churn_prob', color_continuous_scale='Reds',
    title=' Revenue at Risk by Segment (R$)',
    labels={'revenue_at_risk':'Revenue at Risk (BRL)','avg_churn_prob':'Avg Churn Prob'})
fig_risk.update_layout(template='plotly_dark', font_family='Inter')
fig_risk.show()


            Segment  customers    total_clv  revenue_at_risk  avg_churn_prob
    Loyal Customers      27291 4.750143e+06       150.474580        0.000032
   Recent Customers      14980 3.264784e+06       103.421520        0.000032
     Cant Lose Them       8670 2.787832e+06        88.312668        0.000032
          Champions       6492 2.441819e+06        77.351704        0.000032
            At Risk      11150 2.413671e+06        76.460047        0.000032
        Hibernating      11078 2.357876e+06        74.692581        0.000032
Potential Loyalists       4351 1.295880e+06        41.050762        0.000032
               Lost       6315 4.708333e+05        14.915011        0.000032
     Need Attention       3022 2.194731e+05         6.952448        0.000032


## 5. Top 20 Highest-Risk Customers

In [6]:
top20 = df.nlargest(20, 'revenue_at_risk')[
    ['customer_id','Segment','CLV','churn_prob','revenue_at_risk','customer_state']]
print("=== Top 20 Customers by Revenue at Risk ===")
print(top20.to_string(index=False))


=== Top 20 Customers by Revenue at Risk ===
                     customer_id          Segment          CLV  churn_prob  revenue_at_risk customer_state
1617b1357756262bfa56ab541c47bc16   Cant Lose Them 18218.773333    0.000032         0.577133             RJ
ec5b2ba62e574342386871631fafd3fc  Loyal Customers  9699.840000    0.000032         0.307271             ES
c6e2731c5b391845f6800c97401a43a9      Hibernating  9239.080000    0.000032         0.292675             MS
f48d464a0baaea338cb25f816991ab1f Recent Customers  9229.613333    0.000032         0.292375             ES
3fd6777bbce08a352fddd04e4a7cc8f6      Hibernating  8968.880000    0.000032         0.284115             SP
05455dfa7cd02f13d132aa7a6a9729c6   Cant Lose Them  8108.720000    0.000032         0.256867             MG
df55c14d1476a9a3467f131269c2477f              NaN  6600.453333    0.000032         0.209089             RJ
24bbf5fd2f2e1b359ee7de94defc4a15      Hibernating  6352.453333    0.000032         0.201232         

## 6. Revenue at Risk by State — Waterfall

In [7]:
state_risk = df.groupby('customer_state')['revenue_at_risk'].sum().nlargest(8).reset_index()
others = df.groupby('customer_state')['revenue_at_risk'].sum().sort_values(ascending=False)[8:].sum()
state_risk = pd.concat([state_risk,
    pd.DataFrame({'customer_state':['Others'],'revenue_at_risk':[others]})], ignore_index=True)

fig_wf = go.Figure(go.Waterfall(
    name='Revenue at Risk', orientation='v',
    measure=['relative']*len(state_risk),
    x=state_risk['customer_state'],
    y=state_risk['revenue_at_risk'],
    connector={'line':{'color':'rgba(255,255,255,0.3)'}},
    decreasing={'marker':{'color':'#ef4444'}},
    increasing={'marker':{'color':'#10b981'}},
))
fig_wf.update_layout(title='Revenue at Risk by State (Waterfall)',
    template='plotly_dark', font_family='Inter', yaxis_title='BRL')
fig_wf.show()


## 7. Actionable Recommendations

| Segment | Action | Timeline | Expected Impact |
|---------|--------|----------|----------------|
| **Champions** | VIP rewards, exclusive previews, referral program | Ongoing | Protect top-line revenue |
| **At Risk** | Win-back email + 15% discount on preferred category | **Within 7 days** | Recover 20–30% of at-risk customers |
| **Can't Lose Them** | Personal outreach, premium re-engagement offer | **Within 3 days** | High CLV protection |
| **Hibernating** | "We miss you" reactivation campaign | Within 14 days | Reactivate 10–15% |
| **Lost** | Survey + 30% comeback discount | Monthly batch | Low-cost re-engagement |
| **All segments** | Reduce delivery delays >5 days via logistics SLA | Q3 priority | -3pp churn rate per day saved |

> **Delivery Impact**: Simulation shows reducing avg delay by 3 days  500–1,000 fewer churned customers  R$75k–150k revenue recovered.


## 8. Export

In [8]:

# Update EDA summary with revenue figures
try:
    with open('../outputs/eda_summary.json') as f:
        summary = json.load(f)
except FileNotFoundError:
    summary = {}

summary.update({
    'total_clv':          float(total_clv),
    'total_revenue_at_risk': float(total_at_risk),
    'pct_revenue_at_risk':   float(total_at_risk / total_clv),
    'high_risk_customers':   int(high_risk_n),
    'churn_rate':            float(df['churn'].mean()),
})
with open('../outputs/eda_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)
print(f"Revenue at Risk: R$ {total_at_risk:,.0f}")


Revenue at Risk: R$ 654
